<a href="https://colab.research.google.com/github/fabriciosantana/mcdia/blob/main/05-iag/nemotron_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NVIDIA Nemotron - Exemplos de Uso

O **Nemotron** é uma família de modelos da NVIDIA com arquitetura híbrida Mamba-Transformer MoE.

Este notebook mostra como usar o Nemotron via NVIDIA NIM API.

# 1. Importar bibliotecas

In [20]:
import os
from dotenv import load_dotenv

import requests

#from google.colab import userdata


## 1. Configurar API Key


In [21]:
load_dotenv()

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY") # userdata.get('NVIDIA_API_KEY')

if NVIDIA_API_KEY:
    print(f"✅ NVIDIA_API_KEY carregada: {NVIDIA_API_KEY[:15]}...")
else:
    print("⚠️ NVIDIA_API_KEY não encontrada.")

✅ NVIDIA_API_KEY carregada: nvapi-8tdO1NjXD...


## 2. Testar conexão com NVIDIA API

In [22]:

invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "nvidia/nemotron-3-nano-30b-a3b",
    "messages": [
        {"role": "user", "content": "Explique o que é Deep Learning em 3 frases."}
    ],
    "temperature": 0.7,
    "max_tokens": 2000
}

response = requests.post(invoke_url, headers=headers, json=payload)

if response.status_code == 200:
    result = response.json()
    print("✅ Resposta do Nemotron:")
    print(result["choices"][0]["message"]["content"])
else:
    print(f"❌ Erro {response.status_code}: {response.text}")

✅ Resposta do Nemotron:

Deep learning é uma área do aprendizado de máquina que utiliza redes neurais artificiais com múltiplas camadas ocultas para aprender representações hierárquicas de dados. Essas redes são capazes de extrair características cada vez mais complexas, desde pixels brutos até objetos ou padrões semânticos, sem necessidade de engenharia manual das características. Assim, elas podem resolver tarefas como visão computacional, processamento de linguagem natural e reconhecimento de fala com desempenho próximo ou superior ao humano em muitas aplicações.


## 3. Usando a biblioteca OpenAI (compatível)

In [25]:
# pip install openai
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b",
    messages=[
        {"role": "user", "content": "Qual a diferença entre RNN e Transformer?"}
    ],
    temperature=0.7
)

print(response.choices[0].message.content)


## Diferença fundamental entre **RNN** e **Transformer**

| Aspecto | RNN (Recurrent Neural Network) | Transformer |
|---|---|---|
| **Como a informação é processada** | **Passo a passo**: a rede recebe um token de entrada, gera um *hidden state* que contém informação do passado e o passa para o próximo passo. Cada posição depende da anterior. | **Paralelo**: todas as posições são processadas simultaneamente. A dependência entre tokens é modelada por **atenção**, que permite que cada token “olhe” para todos os demais. |
| **Memória implícita** | O *hidden state* funciona como memória que acumula contexto ao longo da sequência. | Não há estado interno; o contexto é representado explicitamente nos vetores de **representação** (embeddings) e nas matrizes de atenção. |
| **Capacidade de longas dependências** | Difícil: o gradiente pode desaparecer ou explodir, dificultando o aprendimento de relações de longo alcance. | Excelente: o mecanismo de atenção captura dependências de longo alcanc

## 4. Streaming (resposta em tempo real)

In [26]:
print("🤖 Nemotron: ", end="")

for chunk in client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b",
    messages=[{"role": "user", "content": "Liste 3 aplicações práticas de LLMs."}],
    temperature=0.7,
    stream=True
):
    if not chunk.choices:
        continue

    choice = chunk.choices[0]
    delta = choice.delta.content
    if delta:
        print(delta, end="", flush=True)
print()

🤖 Nemotron: 
**3 Aplicações práticas de LLMs (Modelos de Linguagem de Grande Escala)**  

| # | Aplicação prática | Como funciona (exemplo) | Benefícios principais |
|---|-------------------|--------------------------|-----------------------|
| 1 | **Assistência e automação de atendimento ao cliente** | Bots conversacionais que entendem a intenção do usuário, respondem a perguntas frequentes, resolvem tickets ou orientam processos (ex.: “Qual o prazo de entrega?” ou “Como trocar minha senha?”). | Redução de custos operacionais, disponibilidade 24/7, experiência mais rápida e personalizada para o cliente. |
| 2 | **Geração e resumo automático de conteúdo** | Modelos criam textos (artigos, posts de blog, roteiros, descrições de produtos) ou condensam documentos longos em resumos executivos. | Economia de tempo de produção, consistência de estilo, possibilidade de escalar conteúdo (ex.: gerar 1.000 descrições de SKU em minutos). |
| 3 | **Análise de texto e extração de insights** | LLMs p

## 5. Chat com contexto (múltiplas mensagens)

In [27]:
messages = [
    {"role": "system", "content": "Você é um especialista em IA. Seja técnico e conciso."},
    {"role": "user", "content": "O que é atenção em transformers?"},
]

response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b",
    messages=messages,
    temperature=0.7
)

print(response.choices[0].message.content)


**Atenção em Transformers – visão técnica resumida**

1. **Motivação**  
   - Substituir arquiteturas recursivas (RNN/CNN) por um mecanismo que processe *todas* as posições simultaneamente.  
   - Permitir que cada token “olhe” (atenda) para todos os outros tokens e aprenda pesos de dependência.

2. **Definição formal (Self‑Attention)**  
   Para um conjunto de *n* tokens, cada um representado por um vetor de embeddings **X** ∈ ℝ^{n×d}:

   - **Projeções lineares** para três matrizes:
     - **Q** = X W_Q  (queries)  
     - **K** = X W_K  (keys)  
     - **V** = X W_V  (values)  
     onde W_Q, W_K, W_V ∈ ℝ^{d×d_k} (geralmente d_k = d_v = d/h).

   - **Cálculo de similaridade** entre cada query e todas as keys:
     \[
     \text{scores}_{i,j}= \frac{Q_i \cdot K_j}{\sqrt{d_k}}
     \]
     (normalização por √d_k estabiliza o gradiente).

   - **Softmax** sobre a dimensão *j* para obter pesos de atenção:
     \[
     \alpha_{i,j}= \text{softmax}_j(\text{scores}_{i,j})
     \]

   - **

## 6. Parâmetros avançados

In [28]:
response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-30b-a3b",
    messages=[{"role": "user", "content": "Gere um título criativo para um artigo sobre IA."}],
    temperature=0.9,
    top_p=0.9
)

print(response.choices[0].message.content)


**"Mentes de Silício: Como a Inteligência Artificial Está Redefinindo o Futuro da Humanidade"**


## Modelos Nemotron Disponíveis

| Modelo | Tamanho | Descrição |
|--------|---------|----------|
| `nvidia/nemotron-3-nano-30b-a3b` | 30B (3.5B ativos) | MoE híbrido, reasoning |
| `nvidia/nemotron-4-340b-instruct` | 340B | Modelo grande para tarefas complexas |

### Características:
- **Arquitetura**: Mamba2-Transformer Hybrid MoE
- **Idiomas**: Inglês, Alemão, Espanhol, Francês, Italiano, Japonês
- **Capacidades**: Reasoning, tool calling, chat, código